# MovieLens Exploratory Data Analysis (EDA) — Setup & Data Inspection

## 1. Introduction & Setup

First, let's import the libraries we need for the EDA:
- **pandas** & **numpy**: data manipulation and basic numerical checks.
- **matplotlib** & **seaborn**: visualizations for the upcoming tasks.
- **pathlib**: cross-platform, relative paths so the notebook runs seamlessly on any teammate's machine.

In [8]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Visual style for plots in upcoming tasks
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["font.size"] = 10

print("Libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")

Libraries imported successfully!
Pandas version: 2.3.3


## 2. Data Loading

We resolve the project root dynamically using `pathlib.Path` so there are no hardcoded machine paths (`/Users/...`).
Our raw CSV files are located in `data/raw/`:
- `ratings.csv`: user ratings for movies (0.5 to 5.0 scale).
- `movies.csv`: movie titles and listed genres.
- `tags.csv`: free-text tags applied by users to movies.
- `links.csv`: identifiers to external databases (IMDb and TMDb).

In [9]:
# Project-relative path resolution
current_dir = Path.cwd()
repo_root = current_dir.parent if current_dir.name == "notebooks" else current_dir
raw_data_dir = repo_root / "data" / "raw"

print(f"Loading raw data from: {raw_data_dir}")

# Load all 4 datasets
ratings = pd.read_csv(raw_data_dir / "ratings.csv")
movies = pd.read_csv(raw_data_dir / "movies.csv")
tags = pd.read_csv(raw_data_dir / "tags.csv")
links = pd.read_csv(raw_data_dir / "links.csv")

# Quick confirmation of loaded records
print(f"ratings: {ratings.shape[0]:,} rows x {ratings.shape[1]} columns")
print(f"movies:  {movies.shape[0]:,} rows x {movies.shape[1]} columns")
print(f"tags:    {tags.shape[0]:,} rows x {tags.shape[1]} columns")
print(f"links:   {links.shape[0]:,} rows x {links.shape[1]} columns")

Loading raw data from: /Users/carla/Desktop/master/ADS/ADS-B1/team-B1-recsys/data/raw
ratings: 100,836 rows x 4 columns
movies:  9,742 rows x 3 columns
tags:    3,683 rows x 4 columns
links:   9,742 rows x 3 columns


## 3. Initial Structure of the 4 Datasets

Let's check the schema, column types, and first few rows of each table.

### 3.1 Ratings table
This is our primary table for the recommender system. Each row represents a rating given by a `userId` to a `movieId` at a specific Unix `timestamp`.

In [10]:
print("Ratings Info:")
ratings.info()
ratings.head()

Ratings Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


### 3.2 Movies table
Contains metadata about each movie: the unique `movieId`, the `title` (which includes the release year in parentheses), and pipe-separated `genres`.

In [11]:
print("Movies Info:")
movies.info()
movies.head()

Movies Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  9742 non-null   int64 
 1   title    9742 non-null   object
 2   genres   9742 non-null   object
dtypes: int64(1), object(2)
memory usage: 228.5+ KB


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


### 3.3 Tags table
Contains user-generated keyword tags. Useful to understand if tags can be leveraged for content-based features or cold-start recommendations.

In [12]:
print("Tags Info:")
tags.info()
tags.head()

Tags Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3683 entries, 0 to 3682
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   userId     3683 non-null   int64 
 1   movieId    3683 non-null   int64 
 2   tag        3683 non-null   object
 3   timestamp  3683 non-null   int64 
dtypes: int64(3), object(1)
memory usage: 115.2+ KB


,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,Highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,Boxing story,1445715207
4,2,89774,MMA,1445715200


### 3.4 Links table
Contains external identifiers (`imdbId`, `tmdbId`) linked to `movieId`. Useful if we later need to enrich movies with posters, synopses, or external metadata.

In [13]:
print("Links Info:")
links.info()
links.head()

Links Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   movieId  9742 non-null   int64  
 1   imdbId   9742 non-null   int64  
 2   tmdbId   9734 non-null   float64
dtypes: float64(1), int64(2)
memory usage: 228.5 KB


,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


### 3.5 Quick Identifier Sanity Check
Before wrapping up Task 1, let's verify the unique counts of primary keys (`userId`, `movieId`) across tables.

In [14]:
print(f"Unique users in ratings:  {ratings['userId'].nunique():,}")
print(f"Unique movies in ratings: {ratings['movieId'].nunique():,}")
print(f"Total movies in catalog:  {movies['movieId'].nunique():,}")
print(f"Unique users who tagged:  {tags['userId'].nunique():,}")
print(f"Unique tags applied:      {tags['tag'].str.lower().nunique():,}")

Unique users in ratings:  610
Unique movies in ratings: 9,724
Total movies in catalog:  9,742
Unique users who tagged:  58
Unique tags applied:      1,475
